In [ ]:
# Confirm GPU is connected
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.stdout else "No GPU found!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# Clone segmentation branch. Skip if it already exists so the cell is rerunnable.
if [ ! -d /content/CV-Assignment2-Group6/.git ]; then
  git clone -b segmentation https://github.com/linenmin/CV-Assignment2-Group6.git /content/CV-Assignment2-Group6
else
  echo "Repo already exists, skip clone."
fi

# Link dataset from Drive. Skip if the link/directory already exists.
if [ ! -e /content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026 ]; then
  ln -s /content/drive/MyDrive/kul-computer-vision-ga-2-2026 \
        /content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026
else
  echo "Dataset link already exists, skip symlink."
fi

echo "Train images: $(ls /content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026/train/img | wc -l)"
echo "Test images: $(ls /content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026/test/img | wc -l)"

In [ ]:
%%bash
# Known-good Colab combo used by the previous segmentation runs.
pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121 -q
pip install mmcv==2.2.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html -q
pip install mmsegmentation==1.2.2 -q

# Relax mmseg's upper-bound check for mmcv 2.2.0 on Colab Python 3.12.
sed -i "s/MMCV_MAX = '2.2.0'/MMCV_MAX = '2.3.0'/g" /usr/local/lib/python3.12/dist-packages/mmseg/__init__.py

pip install -q --upgrade setuptools timm ftfy regex

cd "/content/CV-Assignment2-Group6/Semantic segmentation"
pip install -q -e .

In [ ]:
import torch, mmcv, mmseg
print(f'PyTorch: {torch.__version__}')
print(f'mmcv: {mmcv.__version__}')
print(f'mmsegmentation: {mmseg.__version__}')
print('All OK! Ready to train V6 SegFormer-B2.')

In [ ]:
%%bash
cat > "/content/CV-Assignment2-Group6/Semantic segmentation/configs/experiments/segformer_b2_512x512_adamw_poly_v6.py" << 'EOF'
custom_imports = dict(
    imports=[
        "ga2_seg.early_stopping",
        "ga2_seg.mmseg_dataset",
        "ga2_seg.mmseg_transforms",
    ],
    allow_failed_imports=False,
)

_base_ = [
    "../dataset/ga2_voc_like.py",
    "../runtime/default_runtime.py",
]

checkpoint = "https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b2_20220624-66e8bf70.pth"
data_root = '/content/CV-Assignment2-Group6/kul-computer-vision-ga-2-2026'
project_root = '/content/CV-Assignment2-Group6/Semantic segmentation'

train_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        ann_file=f'{project_root}/data/splits/train.txt',
        data_prefix=dict(
            img_path=f'{data_root}/train/img',
            seg_map_path=f'{data_root}/train/seg',
        )
    )
)
val_dataloader = dict(
    dataset=dict(
        data_root=data_root,
        ann_file=f'{project_root}/data/splits/val.txt',
        data_prefix=dict(
            img_path=f'{data_root}/train/img',
            seg_map_path=f'{data_root}/train/seg',
        )
    )
)
test_dataloader = val_dataloader

model = dict(
    type="EncoderDecoder",
    data_preprocessor=dict(
        type="SegDataPreProcessor",
        mean=[123.675, 116.28, 103.53],
        std=[58.395, 57.12, 57.375],
        bgr_to_rgb=False,
        pad_val=0,
        seg_pad_val=0,
        size=(512, 512),
    ),
    pretrained=None,
    backbone=dict(
        type="MixVisionTransformer",
        in_channels=3,
        embed_dims=64,
        num_stages=4,
        num_layers=[3, 4, 6, 3],
        num_heads=[1, 2, 5, 8],
        patch_sizes=[7, 3, 3, 3],
        sr_ratios=[8, 4, 2, 1],
        out_indices=(0, 1, 2, 3),
        mlp_ratio=4,
        qkv_bias=True,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.1,
        norm_cfg=dict(type="LN", eps=1e-6),
        init_cfg=dict(type="Pretrained", checkpoint=checkpoint),
    ),
    decode_head=dict(
        type="SegformerHead",
        in_channels=[64, 128, 320, 512],
        in_index=[0, 1, 2, 3],
        channels=256,
        dropout_ratio=0.1,
        num_classes=21,
        norm_cfg=dict(type="BN", requires_grad=True),
        align_corners=False,
        loss_decode=dict(type="CrossEntropyLoss", use_sigmoid=False, loss_weight=1.0),
    ),
    train_cfg=dict(),
    test_cfg=dict(mode="whole"),
)

train_cfg = dict(type="IterBasedTrainLoop", max_iters=30000, val_interval=1000)
val_cfg = dict(type="ValLoop")
test_cfg = dict(type="TestLoop")

optim_wrapper = dict(
    type="AmpOptimWrapper",
    optimizer=dict(type="AdamW", lr=6e-5, betas=(0.9, 0.999), weight_decay=0.01),
    paramwise_cfg=dict(custom_keys={
        "pos_block": dict(decay_mult=0.0),
        "norm": dict(decay_mult=0.0),
        "head": dict(lr_mult=10.0),
    }),
)

param_scheduler = [
    dict(type="LinearLR", start_factor=1e-6, by_epoch=False, begin=0, end=1500),
    dict(type="PolyLR", eta_min=0.0, power=1.0, by_epoch=False, begin=1500, end=30000),
]

randomness = dict(seed=42)

custom_hooks = [
    dict(
        type="DelayedEarlyStoppingHook",
        monitor="mIoU",
        rule="greater",
        min_delta=0.1,
        patience=6,
        begin=5,
        strict=False,
    ),
]

work_dir = "./outputs/logs/exp_v6_segformer_b2"
EOF
echo "V6 SegFormer-B2 config created!"

In [ ]:
%%bash
cd "/content/CV-Assignment2-Group6/Semantic segmentation"
python scripts/train.py \
    --config configs/experiments/segformer_b2_512x512_adamw_poly_v6.py \
    --num-workers 4 --batch-size 2 \
    --work-dir outputs/logs/exp_v6_segformer_b2

In [ ]:
%%bash
cd "/content/CV-Assignment2-Group6/Semantic segmentation"

BEST_CKPT=$(ls outputs/logs/exp_v6_segformer_b2/best_mIoU_*.pth 2>/dev/null | head -1)
echo "Best checkpoint: $BEST_CKPT"
if [ -z "$BEST_CKPT" ]; then
  echo "No best checkpoint found. Stop before prediction."
  exit 1
fi

python scripts/predict_test_segmentation.py \
    --config configs/experiments/segformer_b2_512x512_adamw_poly_v6.py \
    --checkpoint $BEST_CKPT \
    --output-dir outputs/predictions/exp_v6_segformer_b2_test

python - <<'PY'
from pathlib import Path
p = Path('src/ga2_seg/submission.py')
text = p.read_text()
old = 'df.loc[idx, CLASS_NAMES]'
new = 'df.loc[idx, list(CLASS_NAMES)]'
if old in text:
    p.write_text(text.replace(old, new))
    print('submission.py pandas compatibility patch applied')
else:
    print('submission.py already pandas compatible')
PY

python scripts/export_submission.py \
    --prediction-dir outputs/predictions/exp_v6_segformer_b2_test \
    --output-path outputs/submissions/submission_exp_v6_segformer_b2.csv \
    --classification-fill 0

python scripts/analyze_training_run.py \
    --work-dir outputs/logs/exp_v6_segformer_b2

In [ ]:
%%bash
mkdir -p /content/drive/MyDrive/CV_Assignment_Outputs/v6_segformer_b2
cp -r "/content/CV-Assignment2-Group6/Semantic segmentation/outputs" \
      /content/drive/MyDrive/CV_Assignment_Outputs/v6_segformer_b2/
echo "Backup complete!"